In [0]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "urbanmart_bronze_panagiotakopoulos_026";
CREATE WIDGET TEXT silver_schema DEFAULT "urbanmart_silver_panagiotakopoulos_026";
CREATE WIDGET TEXT gold_schema DEFAULT "urbanmart_gold_panagiotakopoulos_026";
CREATE WIDGET TEXT source_path DEFAULT "/Volumes/bootcamp/default/volume";

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :bronze_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :silver_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';

USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:gold_schema);

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dim_date (
  date_key BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY COMMENT 'yyyymmdd; -1 = unknown date',
  full_date DATE,
  day_of_week_num  INT COMMENT '1=Monday .. 7=Sunday',
  day_name STRING,
  is_weekend BOOLEAN,
  day_type STRING  COMMENT 'Weekday / Weekend / Unknown',
  week_of_year INT,
  month_num INT,
  month_name STRING,
  year_month STRING  COMMENT 'yyyy-MM, sortable',
  quarter_num INT,
  quarter_name STRING,
  year_num INT,
  CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
) USING DELTA COMMENT 'Calendar dimension';

CREATE TABLE IF NOT EXISTS dim_customer (
  customer_key BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY COMMENT 'Surrogate key; -1 = unknown',
  customer_id  STRING NOT NULL COMMENT 'Natural key',
  first_name STRING, last_name STRING, full_name STRING, email STRING, gender STRING,
  birth_date DATE, city STRING, region STRING, segment STRING, signup_date DATE,
  CONSTRAINT pk_dim_customer PRIMARY KEY (customer_key)
) USING DELTA COMMENT 'Customer dimension (SCD1: latest effective customer_changes applied)';

CREATE TABLE IF NOT EXISTS dim_product (
  product_key BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY COMMENT 'Surrogate key; -1 = unknown',
  product_id STRING NOT NULL COMMENT 'Natural key',
  product_name STRING, category STRING, subcategory STRING, brand STRING,
  unit_cost DECIMAL(12,2), list_price DECIMAL(12,2), is_active BOOLEAN,
  CONSTRAINT pk_dim_product PRIMARY KEY (product_key)
) USING DELTA COMMENT 'Product dimension';

CREATE TABLE IF NOT EXISTS dim_store (
  store_key BIGINT NOT NULL GENERATED BY DEFAULT AS IDENTITY COMMENT 'Surrogate key; -1 = unknown',
  store_id STRING NOT NULL COMMENT 'Natural key',
  store_name STRING, store_type STRING, city STRING, region STRING, open_date DATE, size_sqm INT,
  is_online BOOLEAN COMMENT 'Derived from store_type',
  channel STRING  COMMENT 'Online / Physical / Unknown',
  CONSTRAINT pk_dim_store PRIMARY KEY (store_key)
) USING DELTA COMMENT 'Store dimension';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS fact_sales (
  order_line_id STRING NOT NULL COMMENT 'Degenerate dimension / grain key',
  order_id STRING NOT NULL COMMENT 'Degenerate dimension',
  date_key BIGINT NOT NULL,
  customer_key BIGINT NOT NULL,
  product_key BIGINT NOT NULL,
  store_key BIGINT NOT NULL,
  order_status STRING NOT NULL COMMENT 'Completed / Cancelled / Returned',
  payment_method  STRING,
  quantity INT NOT NULL,
  unit_price DECIMAL(12,2) NOT NULL,
  discount_pct DECIMAL(5,4) NOT NULL,
  unit_cost DECIMAL(12,2) NOT NULL,
  gross_amount DECIMAL(14,2) NOT NULL COMMENT 'quantity * unit_price',
  discount_amount DECIMAL(14,2) NOT NULL COMMENT 'round(gross * discount_pct, 2)',
  net_amount DECIMAL(14,2) NOT NULL COMMENT 'gross - discount',
  cost_amount DECIMAL(14,2) NOT NULL COMMENT 'quantity * unit_cost',
  margin_amount DECIMAL(14,2) NOT NULL COMMENT 'net - cost',
  CONSTRAINT pk_fact_sales PRIMARY KEY (order_line_id),
  CONSTRAINT fk_fact_date FOREIGN KEY (date_key) REFERENCES dim_date(date_key),
  CONSTRAINT fk_fact_customer FOREIGN KEY (customer_key) REFERENCES dim_customer(customer_key),
  CONSTRAINT fk_fact_product FOREIGN KEY (product_key) REFERENCES dim_product(product_key),
  CONSTRAINT fk_fact_store FOREIGN KEY (store_key) REFERENCES dim_store(store_key)
) USING DELTA
CLUSTER BY (date_key)
COMMENT 'Sales fact. Grain: one row per order line.';

ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_qty_pos;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_price_nonneg;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_cost_nonneg;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_disc_range;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_gross;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_disc_amt;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_net;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_margin;
ALTER TABLE fact_sales DROP CONSTRAINT IF EXISTS chk_status;

ALTER TABLE fact_sales ADD CONSTRAINT chk_qty_pos CHECK (quantity >= 1);
ALTER TABLE fact_sales ADD CONSTRAINT chk_price_nonneg CHECK (unit_price >= 0);
ALTER TABLE fact_sales ADD CONSTRAINT chk_cost_nonneg CHECK (unit_cost >= 0);
ALTER TABLE fact_sales ADD CONSTRAINT chk_disc_range CHECK (discount_pct >= 0 AND discount_pct <= 1);
ALTER TABLE fact_sales ADD CONSTRAINT chk_gross CHECK (gross_amount = quantity * unit_price);
ALTER TABLE fact_sales ADD CONSTRAINT chk_disc_amt CHECK (discount_amount >= 0 AND discount_amount <= gross_amount);
ALTER TABLE fact_sales ADD CONSTRAINT chk_net CHECK (net_amount = gross_amount - discount_amount);
ALTER TABLE fact_sales ADD CONSTRAINT chk_margin CHECK (margin_amount = net_amount - cost_amount);
ALTER TABLE fact_sales ADD CONSTRAINT chk_status CHECK (order_status IN ('Completed','Cancelled','Returned'));

In [0]:
%sql
INSERT INTO dim_date (date_key, full_date, day_of_week_num, day_name, is_weekend, day_type, week_of_year,
                      month_num, month_name, year_month, quarter_num, quarter_name, year_num)
SELECT CAST(date_format(d, 'yyyyMMdd') AS BIGINT), d, weekday(d) + 1, date_format(d, 'EEEE'), weekday(d) >= 5,
       CASE WHEN weekday(d) >= 5 THEN 'Weekend' ELSE 'Weekday' END, weekofyear(d),
       month(d), date_format(d, 'MMMM'), date_format(d, 'yyyy-MM'), quarter(d), concat('Q', quarter(d)), year(d)
FROM (
  SELECT explode(sequence(make_date(year(d0), 1, 1), make_date(year(d1), 12, 31), INTERVAL 1 DAY)) AS d
  FROM (SELECT COALESCE(MIN(order_date), DATE'2020-01-01') AS d0, COALESCE(MAX(order_date), current_date()) AS d1
        FROM IDENTIFIER(:silver_schema || '.orders'))
) days
WHERE NOT EXISTS (SELECT 1 FROM dim_date x WHERE x.date_key = CAST(date_format(days.d, 'yyyyMMdd') AS BIGINT));

INSERT INTO dim_date (date_key, full_date, day_name, day_type, month_name, year_month, quarter_name)
SELECT -1, NULL, 'Unknown', 'Unknown', 'Unknown', 'Unknown', 'Unknown'
WHERE NOT EXISTS (SELECT 1 FROM dim_date WHERE date_key = -1);



num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
MERGE INTO dim_customer t
USING (
  SELECT c.customer_id, c.first_name, c.last_name, concat_ws(' ', c.first_name, c.last_name) AS full_name,
         c.email, c.gender, c.birth_date,
         COALESCE(ch.city, c.city) AS city,
         COALESCE(ch.region, c.region) AS region,
         COALESCE(ch.segment, c.segment) AS segment,
         c.signup_date
  FROM IDENTIFIER(:silver_schema || '.customers') c
  LEFT JOIN (
    SELECT customer_id,
           max_by(new_city, effective_date) FILTER (WHERE new_city IS NOT NULL) AS city,
           max_by(new_region, effective_date) FILTER (WHERE new_region IS NOT NULL) AS region,
           max_by(new_segment, effective_date) FILTER (WHERE new_segment IS NOT NULL) AS segment
    FROM IDENTIFIER(:silver_schema || '.customer_changes')
    WHERE effective_date <= current_date()
    GROUP BY customer_id
  ) ch ON ch.customer_id = c.customer_id
) s
ON t.customer_id = s.customer_id
WHEN MATCHED THEN UPDATE SET
  t.first_name = s.first_name, t.last_name = s.last_name, t.full_name = s.full_name, t.email = s.email,
  t.gender = s.gender, t.birth_date = s.birth_date, t.city = s.city, t.region = s.region,
  t.segment = s.segment, t.signup_date = s.signup_date
WHEN NOT MATCHED THEN INSERT (customer_id, first_name, last_name, full_name, email, gender, birth_date, city, region, segment, signup_date)
  VALUES (s.customer_id, s.first_name, s.last_name, s.full_name, s.email, s.gender, s.birth_date, s.city, s.region, s.segment, s.signup_date);

INSERT INTO dim_customer (customer_key, customer_id, first_name, last_name, full_name, email, gender, birth_date, city, region, segment, signup_date)
SELECT -1, 'UNKNOWN', 'Unknown', 'Unknown', 'Unknown', NULL, 'Unknown', NULL, 'Unknown', 'Unknown', 'Unknown', NULL
WHERE NOT EXISTS (SELECT 1 FROM dim_customer WHERE customer_key = -1);



num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
MERGE INTO dim_product t
USING (SELECT product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active
       FROM IDENTIFIER(:silver_schema || '.products')) s
ON t.product_id = s.product_id
WHEN MATCHED THEN UPDATE SET
  t.product_name = s.product_name, t.category = s.category, t.subcategory = s.subcategory, t.brand = s.brand,
  t.unit_cost = s.unit_cost, t.list_price = s.list_price, t.is_active = s.is_active
WHEN NOT MATCHED THEN INSERT (product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active)
  VALUES (s.product_id, s.product_name, s.category, s.subcategory, s.brand, s.unit_cost, s.list_price, s.is_active);

INSERT INTO dim_product (product_key, product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active)
SELECT -1, 'UNKNOWN', 'Unknown', 'Unknown', 'Unknown', 'Unknown', NULL, NULL, NULL
WHERE NOT EXISTS (SELECT 1 FROM dim_product WHERE product_key = -1);



num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
MERGE INTO dim_store t
USING (SELECT store_id, store_name, store_type, city, region, open_date, size_sqm,
              LOWER(store_type) RLIKE 'online|web|e-?com|digital' AS is_online,
              CASE WHEN store_type = 'Unknown' THEN 'Unknown'
                   WHEN LOWER(store_type) RLIKE 'online|web|e-?com|digital' THEN 'Online' ELSE 'Physical' END AS channel
       FROM IDENTIFIER(:silver_schema || '.stores')) s
ON t.store_id = s.store_id
WHEN MATCHED THEN UPDATE SET
  t.store_name = s.store_name, t.store_type = s.store_type, t.city = s.city, t.region = s.region,
  t.open_date = s.open_date, t.size_sqm = s.size_sqm, t.is_online = s.is_online, t.channel = s.channel
WHEN NOT MATCHED THEN INSERT (store_id, store_name, store_type, city, region, open_date, size_sqm, is_online, channel)
  VALUES (s.store_id, s.store_name, s.store_type, s.city, s.region, s.open_date, s.size_sqm, s.is_online, s.channel);

INSERT INTO dim_store (store_key, store_id, store_name, store_type, city, region, open_date, size_sqm, is_online, channel)
SELECT -1, 'UNKNOWN', 'Unknown', 'Unknown', 'Unknown', 'Unknown', NULL, NULL, NULL, 'Unknown'
WHERE NOT EXISTS (SELECT 1 FROM dim_store WHERE store_key = -1);



num_affected_rows,num_inserted_rows
0,0


In [0]:
%sql
INSERT OVERWRITE fact_sales
WITH base AS (
  SELECT i.order_line_id, i.order_id,
         COALESCE(d.date_key, -1)     AS date_key,
         COALESCE(c.customer_key, -1) AS customer_key,
         COALESCE(p.product_key, -1)  AS product_key,
         COALESCE(s.store_key, -1)    AS store_key,
         o.order_status, o.payment_method,
         i.quantity, i.unit_price, i.discount_pct,
         COALESCE(p.unit_cost, CAST(0 AS DECIMAL(12,2))) AS unit_cost,
         CAST(i.quantity * i.unit_price AS DECIMAL(14,2)) AS gross_amount,
         CAST(ROUND(i.quantity * i.unit_price * i.discount_pct, 2) AS DECIMAL(14,2)) AS discount_amount,
         CAST(i.quantity * COALESCE(p.unit_cost, 0) AS DECIMAL(14,2)) AS cost_amount
  FROM IDENTIFIER(:silver_schema || '.order_items') i
  JOIN IDENTIFIER(:silver_schema || '.orders') o ON o.order_id = i.order_id
  LEFT JOIN dim_date d ON d.full_date = o.order_date
  LEFT JOIN dim_customer c ON c.customer_id = o.customer_id
  LEFT JOIN dim_product p ON p.product_id  = i.product_id
  LEFT JOIN dim_store s ON s.store_id = o.store_id
)
SELECT order_line_id, order_id, date_key, customer_key, product_key, store_key, order_status, payment_method,
       quantity, unit_price, discount_pct, unit_cost,
       gross_amount, discount_amount,
       CAST(gross_amount - discount_amount AS DECIMAL(14,2)) AS net_amount,
       cost_amount,
       CAST(gross_amount - discount_amount - cost_amount AS DECIMAL(14,2)) AS margin_amount
FROM base;

-- COMMAND ----------
OPTIMIZE fact_sales;
SELECT 'dim_date' t, COUNT(*) n FROM dim_date UNION ALL SELECT 'dim_customer', COUNT(*) FROM dim_customer
UNION ALL SELECT 'dim_product', COUNT(*) FROM dim_product UNION ALL SELECT 'dim_store', COUNT(*) FROM dim_store
UNION ALL SELECT 'fact_sales', COUNT(*) FROM fact_sales;

t,n
dim_date,732
dim_customer,501
dim_product,77
dim_store,14
fact_sales,23996


In [0]:
%sql
-- COMMAND ----------
-- Q1 Monthly sales performance by month, region, store
SELECT d.year_month, s.region, s.store_name,
       SUM(f.net_amount) AS revenue,
       SUM(f.margin_amount) AS gross_margin,
       ROUND(100 * SUM(f.margin_amount) / NULLIF(SUM(f.net_amount), 0), 1) AS margin_pct,
       SUM(f.quantity) AS units_sold
FROM fact_sales f
JOIN dim_date  d ON f.date_key  = d.date_key
JOIN dim_store s ON f.store_key = s.store_key
WHERE f.order_status = 'Completed'
GROUP BY d.year_month, s.region, s.store_name
ORDER BY d.year_month, revenue DESC;



year_month,region,store_name,revenue,gross_margin,margin_pct,units_sold
2024-01,Attica,UrbanMart Piraeus 2,41820.46,12148.47,29.0,238
2024-01,Attica,UrbanMart Athens 1,41367.93,12208.64,29.5,238
2024-01,Crete,UrbanMart Heraklion 5,36637.09,11210.29,30.6,211
2024-01,Attica,UrbanMart Athens 8,34877.52,10206.38,29.3,218
2024-01,Attica,UrbanMart Piraeus 9,34852.54,10365.90,29.7,201
2024-01,Thessaly,UrbanMart Larissa 6,33348.82,10039.44,30.1,232
2024-01,Attica,UrbanMart Online,32921.43,9316.94,28.3,193
2024-01,Crete,UrbanMart Heraklion 12,29841.04,8941.63,30.0,191
2024-01,Epirus,UrbanMart Ioannina 7,27149.74,8237.57,30.3,162
2024-01,Western Greece,UrbanMart Patras 11,26604.14,7512.37,28.2,172


In [0]:
%sql
SELECT p.category, p.brand,
       SUM(f.net_amount)    AS revenue,
       SUM(f.margin_amount) AS gross_margin,
       ROUND(100 * SUM(f.margin_amount) / NULLIF(SUM(f.net_amount), 0), 1) AS margin_pct,
       RANK() OVER (ORDER BY SUM(f.margin_amount) DESC) AS margin_rank
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
WHERE f.order_status = 'Completed'
GROUP BY p.category, p.brand
ORDER BY gross_margin DESC;



category,brand,revenue,gross_margin,margin_pct,margin_rank
Electronics,Nova,691435.68,229128.48,33.1,1
Grocery,Olympus,562102.06,220879.27,39.3,2
Sports,Olympus,652255.38,211084.74,32.4,3
Grocery,Nova,674323.45,199841.92,29.6,4
Home,Aegean,556413.46,175262.62,31.5,5
Home,Nova,484693.17,150761.93,31.1,6
Electronics,Aegean,588573.13,143041.81,24.3,7
Electronics,Helios,434485.97,122360.56,28.2,8
Apparel,Zephyr,412444.19,115695.63,28.1,9
Apparel,Helios,368175.80,110575.48,30.0,10


In [0]:
%sql
SELECT segment, customer_rank, customer_id, full_name, revenue, gross_margin, orders
FROM (
  SELECT c.segment, c.customer_id, c.full_name,
         SUM(f.net_amount) AS revenue,
         SUM(f.margin_amount) AS gross_margin,
         COUNT(DISTINCT f.order_id) AS orders,
         ROW_NUMBER() OVER (PARTITION BY c.segment ORDER BY SUM(f.net_amount) DESC) AS customer_rank
  FROM fact_sales f
  JOIN dim_customer c ON f.customer_key = c.customer_key
  WHERE f.order_status = 'Completed' AND f.customer_key <> -1
  GROUP BY c.segment, c.customer_id, c.full_name
)
WHERE customer_rank <= 5
ORDER BY segment, customer_rank;



segment,customer_rank,customer_id,full_name,revenue,gross_margin,orders
Loyalty Gold,1,C0065,Giorgos Pappas,37217.82,11459.82,24
Loyalty Gold,2,C0345,Sofia Ioannou,33204.49,8988.41,17
Loyalty Gold,3,C0073,Nikos Georgiou,31291.02,9705.86,25
Loyalty Gold,4,C0225,Dimitris Ioannou,28942.73,8107.10,18
Loyalty Gold,5,C0126,Maria Vasileiou,28246.13,8022.66,22
Loyalty Silver,1,C0043,Dimitris Oikonomou,32752.52,9813.76,23
Loyalty Silver,2,C0462,Nikos Vasileiou,29614.59,9007.21,22
Loyalty Silver,3,C0411,Sofia Georgiou,28718.72,8432.86,19
Loyalty Silver,4,C0104,Maria Ioannou,28651.28,9194.71,19
Loyalty Silver,5,C0422,Nikos Georgiou,27970.37,8987.75,19


In [0]:
%sql

SELECT p.category, s.store_type,
       COUNT_IF(f.order_status = 'Returned') AS returned_lines,
       COUNT(*) AS valid_lines,
       ROUND(100 * COUNT_IF(f.order_status = 'Returned') / COUNT(*), 2) AS return_rate_lines_pct,
       ROUND(100 * SUM(CASE WHEN f.order_status = 'Returned' THEN f.net_amount ELSE 0 END)
                 / NULLIF(SUM(f.net_amount), 0), 2) AS return_rate_value_pct
FROM fact_sales f
JOIN dim_product p ON f.product_key = p.product_key
JOIN dim_store   s ON f.store_key   = s.store_key
WHERE f.order_status <> 'Cancelled'
GROUP BY p.category, s.store_type
ORDER BY return_rate_lines_pct DESC;



category,store_type,returned_lines,valid_lines,return_rate_lines_pct,return_rate_value_pct
Electronics,Mall,101,1397,7.23,6.71
Home,High Street,102,1444,7.06,6.28
Electronics,Online,26,369,7.05,6.45
Home,Mall,96,1408,6.82,6.49
Grocery,Mall,95,1407,6.75,7.08
Apparel,High Street,94,1441,6.52,7.25
Grocery,High Street,91,1476,6.17,5.71
Sports,High Street,89,1488,5.98,6.60
Sports,Mall,84,1427,5.89,6.31
Electronics,High Street,86,1475,5.83,6.49


In [0]:
%sql
SELECT d.day_type, s.channel,
       SUM(f.net_amount) AS revenue,
       COUNT(DISTINCT f.order_id) AS orders,
       ROUND(SUM(f.net_amount) / COUNT(DISTINCT f.order_id), 2) AS avg_order_value,
       COUNT(DISTINCT d.date_key) AS active_days,
       ROUND(SUM(f.net_amount) / COUNT(DISTINCT d.date_key), 2) AS revenue_per_day,
       ROUND(100 * SUM(f.margin_amount) / NULLIF(SUM(f.net_amount), 0), 1) AS margin_pct
FROM fact_sales f
JOIN dim_date  d ON f.date_key  = d.date_key
JOIN dim_store s ON f.store_key = s.store_key
WHERE f.order_status = 'Completed' AND d.date_key <> -1
GROUP BY d.day_type, s.channel
ORDER BY d.day_type, s.channel;

day_type,channel,revenue,orders,avg_order_value,active_days,revenue_per_day,margin_pct
Weekday,Online,529490.05,413,1282.06,287,1844.91,28.7
Weekday,Physical,5999633.61,4725,1269.76,522,11493.55,29.5
Weekend,Online,216087.80,172,1256.32,121,1785.85,30.4
Weekend,Physical,2393143.35,1931,1239.33,208,11505.50,29.3


In [0]:
%sql
select * from bootcamp.urbanmart_bronze_panagiotakopoulos_026.dq_log

id,source_table,column_name,issue_description,row_affected,handling_rule,profiled_at
140,customers,birth_date,Invalid value: expected DATE (yyyy-MM-dd) type,0,DEFAULT: set to NULL (an order date maps to unknown date -1),2026-10-26T16:17:08.656Z
141,customers,signup_date,Invalid value: expected DATE (yyyy-MM-dd) type,0,DEFAULT: set to NULL (an order date maps to unknown date -1),2026-10-26T16:17:08.656Z
142,stores,open_date,Invalid value: expected DATE (yyyy-MM-dd) type,0,DEFAULT: set to NULL (an order date maps to unknown date -1),2026-10-26T16:17:08.656Z
143,stores,size_sqm,Invalid value: expected INT type,0,DEFAULT: set to NULL (an order date maps to unknown date -1),2026-10-26T16:17:08.656Z
144,orders,order_ts,Invalid value: expected TIMESTAMP (yyyy-MM-dd HH:mm:ss or dd/MM/yyyy HH:mm) type,0,DEFAULT: set to NULL (an order date maps to unknown date -1),2026-10-26T16:17:08.656Z
145,order_items,discount_pct,Invalid value: expected DOUBLE type,0,REJECT: invalid discount value,2026-10-26T16:17:08.656Z
151,customers,customer_id,Rows in a customer_id group with conflicting data,0,REJECT: all rows of the key rejected as CONFLICT_KEY,2026-10-26T16:17:17.774Z
152,products,product_id,Rows in a product_id group with conflicting data,0,REJECT: all rows of the key rejected as CONFLICT_KEY,2026-10-26T16:17:17.774Z
153,stores,store_id,Rows in a store_id group with conflicting data,0,REJECT: all rows of the key rejected as CONFLICT_KEY,2026-10-26T16:17:17.774Z
154,orders,order_id,Rows in an order_id group with conflicting data,0,REVIEW: resolve conflicts before Silver,2026-10-26T16:17:17.774Z
